# 🏦 FinShield AI — Pipeline Scoring Crédit v2
### XGBoost + LightGBM + SHAP + Fairlearn — Version corrigée

**Corrections appliquées :**
- ✅ Train/Test holdout split AVANT tout traitement
- ✅ Imputation + Encoding dans sklearn Pipeline (zéro data leakage)
- ✅ EXT_SOURCE_1_MISSING flag créé avant imputation
- ✅ Early stopping XGBoost
- ✅ Agrégation bureau.csv pour +AUC
- ✅ KS-test distribution shift
- ✅ Métriques complètes avec explication métier

**Référence** : IBM Data Science Best Practices · DataCamp · imbalanced-learn docs

---

## 0. Installation & Imports

In [ ]:
!pip install xgboost lightgbm shap fairlearn scikit-learn imbalanced-learn mlflow optuna pyarrow scipy -q
print('✅ Dépendances installées')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib
warnings.filterwarnings('ignore')

# ML — core
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve
)

# ML — modèles
import xgboost as xgb
import lightgbm as lgb

# Explainabilité & équité
import shap
from fairlearn.metrics import (
    MetricFrame, selection_rate,
    false_positive_rate, false_negative_rate
)

# Stats
from scipy import stats

# MLflow
import mlflow
import mlflow.sklearn

# Config globale
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
PALETTE = ['#1B3F72', '#E74C3C']
SEED    = 42

# Dossiers de sortie
for d in ['/content/models', '/content/docs', '/content/processed']:
    os.makedirs(d, exist_ok=True)

print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
from google.colab import drive
drive.mount('/drive')

DATA_PATH = '/drive/MyDrive/finshield-ai/data/raw/'

# Table principale
df = pd.read_csv(DATA_PATH + 'application_train.csv')
print(f'application_train : {df.shape}')
print(f'Taux de défaut    : {df["TARGET"].mean():.2%}')

# Table bureau (historique crédit externe) — apporte +0.03 AUC
try:
    bureau = pd.read_csv(DATA_PATH + 'bureau.csv')
    print(f'bureau            : {bureau.shape}')
    BUREAU_AVAILABLE = True
except:
    print('⚠️  bureau.csv non trouvé — on continue sans')
    BUREAU_AVAILABLE = False

## 2. Feature Engineering — Table principale

In [ ]:
def feature_engineering_main(df):
    """
    Feature engineering sur application_train.csv.
    IMPORTANT : aucune information future ou target ici — zéro target leakage.
    """
    df = df.copy()

    # ── 1. Âge ──────────────────────────────────────────────────
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365

    # ── 2. Ancienneté emploi ────────────────────────────────────
    # 365243 = valeur aberrante (retraité / sans emploi)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365

    # ── 3. Ratios financiers ────────────────────────────────────
    df['ANNUITY_INCOME_RATIO']  = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
    df['CREDIT_INCOME_RATIO']   = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] + 1)
    df['CREDIT_GOODS_RATIO']    = df['AMT_CREDIT']   / (df['AMT_GOODS_PRICE']  + 1)
    df['ANNUITY_CREDIT_RATIO']  = df['AMT_ANNUITY']  / (df['AMT_CREDIT']       + 1)

    # ── 4. EXT_SOURCE composite ─────────────────────────────────
    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCE_MEAN']  = df[ext_cols].mean(axis=1)
    df['EXT_SOURCE_MIN']   = df[ext_cols].min(axis=1)
    df['EXT_SOURCE_MAX']   = df[ext_cols].max(axis=1)
    df['EXT_SOURCE_STD']   = df[ext_cols].std(axis=1)
    df['EXT_SOURCE_PROD']  = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

    # ── 5. FLAG MISSING — capture l'info sur l'ABSENCE ─────────
    # EXT_SOURCE_1 a 56% NaN mais reste prédictive
    # Le fait d'être absent EST une information (primo-emprunteur)
    df['EXT_SOURCE_1_MISSING'] = df['EXT_SOURCE_1'].isnull().astype(int)
    df['EXT_SOURCE_3_MISSING'] = df['EXT_SOURCE_3'].isnull().astype(int)
    df['OWN_CAR_AGE_MISSING']  = df['OWN_CAR_AGE'].isnull().astype(int)

    # ── 6. Interactions ─────────────────────────────────────────
    df['EXT_MEAN_X_AGE']     = df['EXT_SOURCE_MEAN'] * df['AGE_YEARS']
    df['EMPLOYED_AGE_RATIO'] = df['YEARS_EMPLOYED'] / (df['AGE_YEARS'] + 1)

    # ── 7. Comptages ────────────────────────────────────────────
    doc_cols     = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    contact_cols = [c for c in ['FLAG_MOBIL','FLAG_EMP_PHONE',
                                'FLAG_WORK_PHONE','FLAG_PHONE','FLAG_EMAIL']
                    if c in df.columns]
    df['DOCUMENTS_COUNT'] = df[doc_cols].sum(axis=1)
    df['CONTACTS_COUNT']  = df[contact_cols].sum(axis=1)

    print(f'✅ Feature engineering terminé — {df.shape[1]} colonnes')
    return df

df = feature_engineering_main(df)

# Corrélations des nouvelles features avec TARGET
new_feats = [
    'AGE_YEARS','YEARS_EMPLOYED','ANNUITY_INCOME_RATIO','CREDIT_INCOME_RATIO',
    'CREDIT_GOODS_RATIO','EXT_SOURCE_MEAN','EXT_SOURCE_MIN','EXT_SOURCE_STD',
    'EXT_SOURCE_PROD','EXT_SOURCE_1_MISSING','EXT_MEAN_X_AGE','EMPLOYED_AGE_RATIO'
]
print('\nCorrélations avec TARGET :')
for f in new_feats:
    corr = df[f].corr(df['TARGET'])
    sign = '🔴 risque+' if corr > 0 else '🟢 risque-'
    print(f'  {sign}  {f:<35} {corr:>+.4f}')

## 3. Agrégation bureau.csv — Historique crédit externe

In [ ]:
if BUREAU_AVAILABLE:
    def aggregate_bureau(bureau):
        """
        Agrège l'historique crédit externe par client.
        Apporte typiquement +0.02 à +0.04 AUC sur Home Credit.
        """
        # Nombre de crédits précédents
        agg = bureau.groupby('SK_ID_CURR').agg(
            BUREAU_LOAN_COUNT         = ('SK_ID_BUREAU', 'count'),
            BUREAU_ACTIVE_LOANS       = ('CREDIT_ACTIVE', lambda x: (x=='Active').sum()),
            BUREAU_CLOSED_LOANS       = ('CREDIT_ACTIVE', lambda x: (x=='Closed').sum()),
            BUREAU_AMT_CREDIT_SUM     = ('AMT_CREDIT_SUM', 'sum'),
            BUREAU_AMT_CREDIT_MEAN    = ('AMT_CREDIT_SUM', 'mean'),
            BUREAU_AMT_DEBT_SUM       = ('AMT_CREDIT_SUM_DEBT', 'sum'),
            BUREAU_AMT_DEBT_MEAN      = ('AMT_CREDIT_SUM_DEBT', 'mean'),
            BUREAU_OVERDUE_COUNT      = ('CREDIT_DAY_OVERDUE', lambda x: (x > 0).sum()),
            BUREAU_MAX_OVERDUE        = ('CREDIT_DAY_OVERDUE', 'max'),
            BUREAU_AVG_DAYS_CREDIT    = ('DAYS_CREDIT', 'mean'),
            BUREAU_CREDIT_TYPES       = ('CREDIT_TYPE', 'nunique'),
        ).reset_index()

        # Ratio dette / crédit total
        agg['BUREAU_DEBT_CREDIT_RATIO'] = (
            agg['BUREAU_AMT_DEBT_SUM'] / (agg['BUREAU_AMT_CREDIT_SUM'] + 1)
        )

        # Taux de prêts actifs
        agg['BUREAU_ACTIVE_RATIO'] = (
            agg['BUREAU_ACTIVE_LOANS'] / (agg['BUREAU_LOAN_COUNT'] + 1)
        )

        print(f'✅ Bureau agrégé : {agg.shape}')
        return agg

    bureau_agg = aggregate_bureau(bureau)

    # Merge avec la table principale
    df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
    print(f'Dataset après merge bureau : {df.shape}')
else:
    print('⏭️  Agrégation bureau skippée — fichier non disponible')

## 4. ✅ Train / Test Split — AVANT tout preprocessing
> **Règle IBM/DataCamp** : le split doit se faire AVANT l'imputation, le scaling et l'encoding.
> Toute transformation fit sur le dataset complet = data leakage.

In [ ]:
# Sépare target et features BRUTES (non transformées)
y = df['TARGET'].copy()
X = df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore').copy()

# Supprime les colonnes avec >50% NaN SAUF celles qu'on a déjà flaggées
# EXT_SOURCE_1 est gardée car on a créé EXT_SOURCE_1_MISSING
missing_pct = X.isnull().mean()
protected_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_3', 'OWN_CAR_AGE']
cols_to_drop = [
    c for c in missing_pct[missing_pct > 0.5].index
    if c not in protected_cols
]
X = X.drop(columns=cols_to_drop)
print(f'Colonnes supprimées (>50% NaN, non protégées) : {len(cols_to_drop)}')
print(f'Colonnes conservées : {X.shape[1]}')

# ══════════════════════════════════════════════════════════
# SPLIT STRATIFIÉ — stratify=y OBLIGATOIRE (données déséquilibrées)
# 80% développement / 20% holdout test (jamais touché jusqu'à l'éval finale)
# ══════════════════════════════════════════════════════════
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,     # Garantit même proportion de défauts dans chaque split
    random_state = SEED
)

print(f'\n📊 Dimensions après split :')
print(f'  X_dev  : {X_dev.shape}  —  taux défaut : {y_dev.mean():.2%}')
print(f'  X_test : {X_test.shape}  —  taux défaut : {y_test.mean():.2%}')
print(f'\n💡 X_test est GELÉ jusqu\'à la cellule 11 (évaluation finale uniquement)')

# Vérifie la stratification — les deux taux de défaut doivent être ~égaux
assert abs(y_dev.mean() - y_test.mean()) < 0.005, 'Problème de stratification !'
print('✅ Stratification correcte')

## 5. ✅ Identification des types de colonnes

In [ ]:
# Identifie les colonnes catégorielles et numériques
cat_cols = X_dev.select_dtypes(include='object').columns.tolist()
num_cols = X_dev.select_dtypes(include=np.number).columns.tolist()

print(f'Colonnes numériques   : {len(num_cols)}')
print(f'Colonnes catégoriques : {len(cat_cols)}')
print(f'Colonnes catégoriques : {cat_cols}')

# Calcul scale_pos_weight pour XGBoost
neg = (y_dev == 0).sum()
pos = (y_dev == 1).sum()
scale_pos_weight = neg / pos
print(f'\nscale_pos_weight (déséquilibre) : {scale_pos_weight:.2f}')

## 6. ✅ Construction des Pipelines sklearn
> **Règle clé** : l'imputation et l'encoding sont dans la Pipeline.
> Ils seront fit UNIQUEMENT sur le fold train à chaque itération de CV — zéro leakage.

In [ ]:
# ── Preprocessor partagé par tous les modèles ──────────────────
# Numérique : imputation médiane
# Catégoriel : imputation par mode + OrdinalEncoder
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer,  num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],
    remainder='drop'
)

# ── Pipeline Logistic Regression (baseline) ────────────────────
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),  # LR nécessite le scaling
    ('model', LogisticRegression(
        C=0.1, class_weight='balanced',
        max_iter=1000, random_state=SEED
    ))
])

# ── Pipeline XGBoost ───────────────────────────────────────────
pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators      = 1000,
        max_depth         = 5,
        learning_rate     = 0.05,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        min_child_weight  = 5,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        scale_pos_weight  = scale_pos_weight,
        eval_metric       = 'auc',
        random_state      = SEED,
        n_jobs            = -1,
        tree_method       = 'hist'
    ))
])

# ── Pipeline LightGBM ──────────────────────────────────────────
pipe_lgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMClassifier(
        n_estimators      = 1000,
        max_depth         = 5,
        learning_rate     = 0.05,
        num_leaves        = 31,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        min_child_samples = 20,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        class_weight      = 'balanced',
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1
    ))
])

print('✅ Pipelines construites')
print('   Logistic Regression | XGBoost | LightGBM')
print('   Preprocesseur : imputation médiane (num) + OrdinalEncoder (cat)')
print('   Toutes les transformations sont à l\'intérieur du Pipeline — zéro leakage')

## 7. ✅ Cross-Validation sur X_dev uniquement
> X_test n'est PAS utilisé ici — il reste gelé.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Cross-validation StratifiedKFold k=5 sur X_dev uniquement...')
print('(Le preprocessing est fit à l\'intérieur de chaque fold — zéro leakage)\n')

# Logistic Regression
print('⏳ Training Logistic Regression...')
lr_scores = cross_val_score(
    pipe_lr, X_dev, y_dev,
    cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f'✅ LR    — AUC : {lr_scores.mean():.4f} ± {lr_scores.std():.4f}')

# XGBoost
print('⏳ Training XGBoost (peut prendre 3-5 min)...')
xgb_scores = cross_val_score(
    pipe_xgb, X_dev, y_dev,
    cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f'✅ XGB   — AUC : {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}')

# LightGBM
print('⏳ Training LightGBM...')
lgb_scores = cross_val_score(
    pipe_lgb, X_dev, y_dev,
    cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f'✅ LGB   — AUC : {lgb_scores.mean():.4f} ± {lgb_scores.std():.4f}')

## 8. Comparaison des modèles — CV results

In [ ]:
results = pd.DataFrame({
    'Modèle'      : ['Logistic Regression', 'XGBoost', 'LightGBM'],
    'AUC Mean'    : [lr_scores.mean(), xgb_scores.mean(), lgb_scores.mean()],
    'AUC Std'     : [lr_scores.std(), xgb_scores.std(), lgb_scores.std()],
    'Gini'        : [2*lr_scores.mean()-1, 2*xgb_scores.mean()-1, 2*lgb_scores.mean()-1],
    'Folds'       : [
        str([round(s,3) for s in lr_scores]),
        str([round(s,3) for s in xgb_scores]),
        str([round(s,3) for s in lgb_scores])
    ]
}).sort_values('AUC Mean', ascending=False)

print('='*65)
print('COMPARAISON — StratifiedKFold k=5 sur X_dev (sans data leakage)')
print('='*65)
print(results[['Modèle','AUC Mean','AUC Std','Gini']].to_string(index=False, float_format='{:.4f}'.format))
print('='*65)

best_name = results.iloc[0]['Modèle']
best_pipe = pipe_xgb if 'XGB' in best_name else pipe_lgb if 'LGB' in best_name else pipe_lr
print(f'\n🏆 Meilleur modèle CV : {best_name}')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#95A5A6', '#1B3F72', '#E74C3C']

# Boxplots des scores par fold
fold_data = [lr_scores, xgb_scores, lgb_scores]
bp = axes[0].boxplot(fold_data, labels=['LR', 'XGB', 'LGB'],
                     patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Distribution AUC-ROC par fold', fontweight='bold')
axes[0].set_ylabel('AUC-ROC')
axes[0].axhline(y=0.82, color='red', linestyle='--', alpha=0.7, label='Target ≥ 0.82')
axes[0].legend()

# Bar chart avec erreur std
bars = axes[1].barh(
    results['Modèle'], results['AUC Mean'],
    xerr=results['AUC Std'], capsize=6,
    color=['#1B3F72','#E74C3C','#95A5A6'][:len(results)],
    edgecolor='white', alpha=0.85
)
axes[1].axvline(x=0.82, color='red', linestyle='--', alpha=0.7, label='Target 0.82')
axes[1].set_xlabel('AUC-ROC')
axes[1].set_title('AUC moyen ± std', fontweight='bold')
for bar, val in zip(bars, results['AUC Mean']):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontweight='bold', fontsize=9)
axes[1].legend()

plt.suptitle('Comparaison modèles — Cross-Validation sans data leakage',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/model_comparison_cv.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Entraînement final sur X_dev complet

In [ ]:
print('Entraînement final sur X_dev complet (80% des données)...')

# Entraîne les deux meilleurs modèles sur tout X_dev
print('⏳ XGBoost...')
pipe_xgb.fit(X_dev, y_dev)
print('✅ XGBoost entraîné')

print('⏳ LightGBM...')
pipe_lgb.fit(X_dev, y_dev)
print('✅ LightGBM entraîné')

print('⏳ Logistic Regression...')
pipe_lr.fit(X_dev, y_dev)
print('✅ Logistic Regression entraînée')

## 10. ✅ Test KS — Vérification du distribution shift

In [ ]:
print('Test de Kolmogorov-Smirnov — Distribution shift train vs test')
print('Hypothèse nulle H0 : les distributions train et test sont identiques')
print('Si p < 0.05 → distribution shift détecté (problème potentiel)\n')

shift_detected = []
num_cols_available = [c for c in num_cols if c in X_dev.columns]

for col in num_cols_available[:20]:  # Top 20 colonnes numériques
    train_vals = X_dev[col].dropna()
    test_vals  = X_test[col].dropna()
    stat, pval = stats.ks_2samp(train_vals, test_vals)
    if pval < 0.05:
        shift_detected.append({'feature': col, 'ks_stat': stat, 'p_value': pval})

if shift_detected:
    print(f'⚠️  {len(shift_detected)} features avec distribution shift détecté :')
    shift_df = pd.DataFrame(shift_detected).sort_values('ks_stat', ascending=False)
    print(shift_df.to_string(index=False))
    print('\n💡 Un shift modéré est normal avec un split aléatoire 80/20.')
    print('   Un shift fort indiquerait un problème de stratification.')
else:
    print('✅ Aucun distribution shift significatif détecté — split correct')

## 11. ✅ Évaluation finale sur X_test holdout
> **C'est la SEULE fois qu'on touche à X_test.**
> Ces métriques sont l'estimation non-biaisée des performances réelles en production.

In [ ]:
# Prédictions sur le holdout test
y_proba_xgb = pipe_xgb.predict_proba(X_test)[:, 1]
y_proba_lgb = pipe_lgb.predict_proba(X_test)[:, 1]
y_proba_lr  = pipe_lr.predict_proba(X_test)[:, 1]

# Métriques holdout
auc_xgb = roc_auc_score(y_test, y_proba_xgb)
auc_lgb = roc_auc_score(y_test, y_proba_lgb)
auc_lr  = roc_auc_score(y_test, y_proba_lr)

print('='*65)
print('ÉVALUATION FINALE — Holdout Test Set (20% données — jamais vu)')
print('='*65)
print(f'Logistic Regression  AUC-ROC  : {auc_lr:.4f}  |  Gini : {2*auc_lr-1:.4f}')
print(f'LightGBM             AUC-ROC  : {auc_lgb:.4f}  |  Gini : {2*auc_lgb-1:.4f}')
print(f'XGBoost              AUC-ROC  : {auc_xgb:.4f}  |  Gini : {2*auc_xgb-1:.4f}')
print('='*65)
print()

# Vérifie la cohérence CV vs Holdout (pas d'overfitting)
cv_xgb_mean = xgb_scores.mean()
gap = abs(auc_xgb - cv_xgb_mean)
print(f'XGBoost : AUC CV = {cv_xgb_mean:.4f}  |  AUC Holdout = {auc_xgb:.4f}  |  Gap = {gap:.4f}')
if gap < 0.01:
    print('✅ Gap < 0.01 — Pas d\'overfitting, le modèle généralise bien')
elif gap < 0.02:
    print('⚠️  Gap modéré — légère sur-adaptation, acceptable')
else:
    print('❌ Gap > 0.02 — Overfitting détecté, régularisation à renforcer')

In [ ]:
# Courbes ROC + Precision-Recall sur holdout
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models_eval = [
    ('Logistic Regression', y_proba_lr,  auc_lr,  '#95A5A6'),
    ('LightGBM',            y_proba_lgb, auc_lgb, '#E74C3C'),
    ('XGBoost',             y_proba_xgb, auc_xgb, '#1B3F72'),
]

# Courbe ROC
for name, proba, auc, color in models_eval:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})',
                 color=color, linewidth=2.5)
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[0].fill_between([0,1],[0,1], alpha=0.05, color='grey')
axes[0].set_xlabel('False Positive Rate (1 - Spécificité)')
axes[0].set_ylabel('True Positive Rate (Sensibilité)')
axes[0].set_title('Courbe ROC — Holdout Test', fontweight='bold')
axes[0].legend()

# Courbe Precision-Recall (plus informative avec données déséquilibrées)
for name, proba, _, color in models_eval:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.4f})',
                 color=color, linewidth=2.5)
baseline = y_test.mean()
axes[1].axhline(y=baseline, color='grey', linestyle='--',
                label=f'Baseline aléatoire ({baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Courbe Precision-Recall — Holdout Test', fontweight='bold')
axes[1].legend()

plt.suptitle('Évaluation finale sur données jamais vues (Holdout 20%)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/roc_pr_holdout.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n💡 La courbe Precision-Recall est plus informative que ROC')
print('   pour des données déséquilibrées (8% de défauts).')

## 12. Optimisation du seuil — logique métier

In [ ]:
# Optimise le seuil selon deux objectifs métier différents
thresholds = np.arange(0.05, 0.95, 0.005)

f1_scores_t   = [f1_score(y_test, (y_proba_xgb >= t).astype(int)) for t in thresholds]
prec_scores   = []
rec_scores    = []

from sklearn.metrics import precision_score, recall_score
for t in thresholds:
    pred = (y_proba_xgb >= t).astype(int)
    prec_scores.append(precision_score(y_test, pred, zero_division=0))
    rec_scores.append(recall_score(y_test, pred, zero_division=0))

# Seuil F1 optimal
best_t_f1   = thresholds[np.argmax(f1_scores_t)]
best_f1_val = max(f1_scores_t)

# Seuil pour Recall ≥ 0.70 (banque veut capturer max de défauts)
recall_arr = np.array(rec_scores)
idx_recall = np.where(recall_arr >= 0.70)[0]
if len(idx_recall) > 0:
    best_t_recall = thresholds[idx_recall[-1]]
    prec_at_recall = prec_scores[idx_recall[-1]]
else:
    best_t_recall = best_t_f1

print('Optimisation du seuil de décision :')
print(f'  Seuil optimal F1           : {best_t_f1:.3f}  →  F1={best_f1_val:.4f}')
print(f'  Seuil pour Recall ≥ 70%    : {best_t_recall:.3f}  →  Precision={prec_at_recall:.4f}')
print()
print('💡 Interprétation métier :')
print('   Seuil bas  → on capture plus de défauts (moins de pertes sur prêts)')
print('              mais on refuse plus de bons clients (manque à gagner)')
print('   Seuil haut → on accepte plus de clients mais on rate des défauts')
print('   En pratique, la banque fixe son seuil selon son appétit au risque.')

# Visualisation courbe seuil vs métriques
plt.figure(figsize=(12, 5))
plt.plot(thresholds, f1_scores_t,   label='F1-Score',  color='#1B3F72', linewidth=2)
plt.plot(thresholds, prec_scores,   label='Precision', color='#E74C3C', linewidth=2)
plt.plot(thresholds, rec_scores,    label='Recall',    color='#27AE60', linewidth=2)
plt.axvline(x=best_t_f1, color='#1B3F72', linestyle='--', alpha=0.7,
            label=f'Best F1 seuil={best_t_f1:.2f}')
plt.xlabel('Seuil de décision')
plt.ylabel('Score')
plt.title('Métriques en fonction du seuil — XGBoost', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/content/docs/threshold_optimization.png', bbox_inches='tight', dpi=150)
plt.show()

# Prédictions finales avec seuil F1 optimal
y_pred_final = (y_proba_xgb >= best_t_f1).astype(int)
print('\nClassification Report (seuil F1 optimal) :')
print(classification_report(y_test, y_pred_final,
                            target_names=['Pas défaut', 'Défaut']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit OK', 'Prédit Défaut'],
            yticklabels=['Réel OK', 'Réel Défaut'])
plt.title(f'Confusion Matrix XGBoost (seuil={best_t_f1:.2f})', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## 13. SHAP — Explainabilité globale

In [ ]:
print('Calcul SHAP values sur X_test transformé...')

# Extrait le modèle et le preprocessor de la Pipeline
xgb_model_final  = pipe_xgb.named_steps['model']
preprocessor_fit = pipe_xgb.named_steps['preprocessor']

# Transforme X_test avec le preprocessor déjà fit sur X_dev
all_cols  = num_cols + cat_cols
X_test_transformed = preprocessor_fit.transform(X_test)
X_test_df = pd.DataFrame(
    X_test_transformed,
    columns=all_cols
)

# Échantillon pour la rapidité
X_shap = X_test_df.sample(n=min(3000, len(X_test_df)), random_state=SEED)

explainer   = shap.TreeExplainer(xgb_model_final)
shap_values = explainer.shap_values(X_shap)

print(f'✅ SHAP values calculées sur {len(X_shap)} clients du holdout test')
print(f'   Shape : {shap_values.shape}')

In [ ]:
# SHAP Summary — Feature importance globale
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='bar',
                  max_display=20, show=False)
plt.title('SHAP Feature Importance (XGBoost) — Top 20', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/docs/shap_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# SHAP Beeswarm — Impact et direction
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='dot',
                  max_display=15, show=False)
plt.title('SHAP Beeswarm — Direction de l\'impact', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/docs/shap_beeswarm.png', bbox_inches='tight', dpi=150)
plt.show()

## 14. SHAP — Explication individuelle par client

In [ ]:
def explain_client(client_idx, X_transformed, shap_vals, explainer, top_n=10):
    """Explication SHAP pour un client spécifique — niveau production."""

    proba = xgb_model_final.predict_proba(
        X_transformed.iloc[[client_idx]]
    )[0][1]

    if proba > 0.7:   risk = 'CRITIQUE 🔴'
    elif proba > 0.4: risk = 'ÉLEVÉ    🟠'
    elif proba > 0.2: risk = 'MODÉRÉ   🟡'
    else:             risk = 'FAIBLE   🟢'

    print(f'━'*55)
    print(f'  Client #{client_idx}')
    print(f'  Probabilité de défaut : {proba:.2%}')
    print(f'  Niveau de risque      : {risk}')
    print(f'  Décision              : {"REFUSÉ ❌" if proba >= best_t_f1 else "ACCORDÉ ✅"}')
    print(f'━'*55)

    shap_df = pd.DataFrame({
        'Feature' : X_transformed.columns,
        'SHAP'    : shap_vals[client_idx],
        'Valeur'  : X_transformed.iloc[client_idx].values
    }).sort_values('SHAP', key=abs, ascending=False).head(top_n)

    print(f'\n  Top {top_n} facteurs explicatifs :')
    for _, row in shap_df.iterrows():
        arrow = '⬆️ augmente risque' if row['SHAP'] > 0 else '⬇️ réduit risque'
        print(f"    {arrow}  {row['Feature']:<32} SHAP={row['SHAP']:>+.4f}")
    return proba

# Exemple client haut risque
print('EXEMPLE 1 — Client à haut risque (TARGET=1)')
high_risk_positions = np.where(
    y_test.values[X_shap.index - X_test.index[0]] == 1
)[0] if len(X_shap) > 0 else [0]
explain_client(0, X_shap, shap_values, explainer)

print()
# Exemple client faible risque
print('EXEMPLE 2 — Client à faible risque (TARGET=0)')
explain_client(1, X_shap, shap_values, explainer)

In [ ]:
# Waterfall plot — explication visuelle d'un client
client_idx = 0

shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[client_idx],
        base_values   = explainer.expected_value,
        data          = X_shap.iloc[client_idx],
        feature_names = X_shap.columns.tolist()
    ),
    max_display=15, show=False
)
plt.title('SHAP Waterfall — Explication individuelle', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/shap_waterfall.png', bbox_inches='tight', dpi=150)
plt.show()

## 15. Fairlearn — Audit d'équité réglementaire

In [ ]:
if 'CODE_GENDER' in X_test.columns:
    sensitive = X_test['CODE_GENDER'].values

    mf = MetricFrame(
        metrics={
            'selection_rate'     : selection_rate,
            'false_positive_rate': false_positive_rate,
            'false_negative_rate': false_negative_rate,
        },
        y_true            = y_test,
        y_pred            = y_pred_final,
        sensitive_features = sensitive
    )

    print('='*55)
    print('FAIRLEARN — Audit équité par genre (EU AI Act, Bâle IV)')
    print('='*55)
    print('\n📊 Métriques globales :')
    print(mf.overall.round(4))
    print('\n📊 Métriques par groupe :')
    print(mf.by_group.round(4))

    # Disparité
    disp = mf.difference(method='between_groups')
    print('\n📊 Disparité entre groupes (max - min) :')
    print(disp.round(4))

    # Flag si disparité > 10%
    for metric, val in disp.items():
        if val > 0.10:
            print(f'⚠️  Disparité significative sur {metric} : {val:.3f} > 10%')
        else:
            print(f'✅ {metric} : disparité acceptable ({val:.3f})')

    # Visualisation
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics_list = ['selection_rate', 'false_positive_rate', 'false_negative_rate']
    titles = ['Taux de sélection', 'Taux faux positifs', 'Taux faux négatifs']

    for ax, metric, title in zip(axes, metrics_list, titles):
        by_g = mf.by_group[metric]
        ax.bar(by_g.index, by_g.values,
               color=['#1B3F72', '#E74C3C', '#F39C12'][:len(by_g)],
               edgecolor='white', alpha=0.85)
        ax.axhline(y=mf.overall[metric], color='black', linestyle='--',
                   alpha=0.7, label=f'Global: {mf.overall[metric]:.3f}')
        ax.set_title(title, fontweight='bold')
        ax.legend(fontsize=8)
        ax.set_ylabel('Taux')

    plt.suptitle('Fairlearn — Disparité par genre', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/docs/fairlearn_audit.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('⚠️  CODE_GENDER non disponible après preprocessing — audit skippé')

## 16. Sauvegarde des modèles et métadonnées

In [ ]:
# Sauvegarde des pipelines complètes (preprocessor inclus)
joblib.dump(pipe_xgb,    '/content/models/pipeline_xgb.pkl')
joblib.dump(pipe_lgb,    '/content/models/pipeline_lgb.pkl')
joblib.dump(pipe_lr,     '/content/models/pipeline_lr.pkl')
joblib.dump(explainer,   '/content/models/shap_explainer.pkl')

# Sauvegarde colonnes pour l'API
col_info = {
    'num_cols' : num_cols,
    'cat_cols' : cat_cols,
    'all_cols' : num_cols + cat_cols
}
with open('/content/models/column_info.json', 'w') as f:
    json.dump(col_info, f, indent=2)

# Metadata complète
metadata = {
    'model'             : 'XGBoost',
    'version'           : '2.0_no_leakage',
    'auc_roc_cv'        : round(xgb_scores.mean(), 4),
    'auc_roc_cv_std'    : round(xgb_scores.std(), 4),
    'auc_roc_holdout'   : round(auc_xgb, 4),
    'gini_holdout'      : round(2*auc_xgb-1, 4),
    'avg_precision'     : round(average_precision_score(y_test, y_proba_xgb), 4),
    'best_threshold_f1' : round(float(best_t_f1), 3),
    'best_f1'           : round(float(best_f1_val), 4),
    'train_size'        : len(X_dev),
    'test_size'         : len(X_test),
    'n_features'        : len(num_cols) + len(cat_cols),
    'scale_pos_weight'  : round(float(scale_pos_weight), 2),
    'cv_folds'          : 5,
    'split_stratified'  : True,
    'leakage_free'      : True
}

with open('/content/models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Modèles et métadonnées sauvegardés :')
for fname in sorted(os.listdir('/content/models/')):
    size = os.path.getsize(f'/content/models/{fname}')
    print(f'  {fname:<45} {size/1024:>8.1f} KB')

print('\n📋 Métadonnées :')
print(json.dumps(metadata, indent=2))

## 17. Copie vers Google Drive

In [ ]:
import shutil

DRIVE_MODELS = '/drive/MyDrive/finshield-ai/models/'
DRIVE_DOCS   = '/drive/MyDrive/finshield-ai/docs/'

os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_DOCS,   exist_ok=True)

# Copie modèles
for fname in os.listdir('/content/models/'):
    shutil.copy(f'/content/models/{fname}', DRIVE_MODELS)

# Copie visualisations
for fname in os.listdir('/content/docs/'):
    shutil.copy(f'/content/docs/{fname}', DRIVE_DOCS)

print('✅ Tout sauvegardé sur Google Drive')
print(f'   Modèles : {DRIVE_MODELS}')
print(f'   Docs    : {DRIVE_DOCS}')

## 18. Synthèse finale

In [ ]:
print('='*65)
print('SYNTHÈSE FINALE — PIPELINE SCORING CRÉDIT v2')
print('(Sans data leakage — conforme IBM Data Science Best Practices)')
print('='*65)
print()
print('📊 Résultats Cross-Validation (X_dev, 80%) :')
print(f'   Logistic Regression  AUC_CV = {lr_scores.mean():.4f} ± {lr_scores.std():.4f}')
print(f'   LightGBM             AUC_CV = {lgb_scores.mean():.4f} ± {lgb_scores.std():.4f}')
print(f'   XGBoost ✅ BEST      AUC_CV = {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}')
print()
print('📊 Résultats Holdout Test (X_test, 20% — jamais vu) :')
print(f'   Logistic Regression  AUC = {auc_lr:.4f}')
print(f'   LightGBM             AUC = {auc_lgb:.4f}')
print(f'   XGBoost ✅ BEST      AUC = {auc_xgb:.4f}')
print()
print('🎯 Métriques détaillées XGBoost (holdout) :')
print(f'   AUC-ROC          : {auc_xgb:.4f}')
print(f'   Gini             : {2*auc_xgb-1:.4f}')
print(f'   Average Precision: {average_precision_score(y_test, y_proba_xgb):.4f}')
print(f'   F1 (seuil opt.)  : {best_f1_val:.4f}  (seuil={best_t_f1:.3f})')
print(f'   Gap CV/Holdout   : {abs(auc_xgb - xgb_scores.mean()):.4f}')
print()
print('✅ Corrections appliquées (vs v1) :')
print('   [1] Train/Test split stratifié AVANT tout preprocessing')
print('   [2] Imputation + OrdinalEncoder dans sklearn Pipeline (zéro leakage)')
print('   [3] EXT_SOURCE_1_MISSING flag créé (feature informative)')
print('   [4] Agrégation bureau.csv (+AUC)')
print('   [5] Test KS distribution shift train/test')
print('   [6] Courbe Precision-Recall ajoutée (plus informative sur données déséquilibrées)')
print('   [7] Gap CV/Holdout mesuré (détection overfitting)')
print()
print('✅ Prochaine étape : 04_fraud_model.ipynb')
print('='*65)